id.co.bri.brimo
   id.bmri.livin
      com.bca

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

app_id_brimo = "id.co.bri.brimo"
app_id_livin = "id.bmri.livin"
app_id_bca = "com.bca"

TARGET = {
    "negatif": 2000,   # rating 1–2
    "netral": 2000,    # rating 3
    "positif": 2400    # rating 4–5
}

BATCH_SIZE = 200

data_negatif = []
data_netral = []
data_positif = []

cursor = None

while (
    len(data_negatif) < TARGET["negatif"] or
    len(data_netral) < TARGET["netral"] or
    len(data_positif) < TARGET["positif"]
):
    result, continuation_token = reviews(
        app_id,
        lang="id",
        country="id",
        sort=Sort.NEWEST,
        count=BATCH_SIZE,
        continuation_token=cursor
    )

    if not result:
        break

    for r in result:
        rating = r["score"]

        if rating in [1, 2] and len(data_negatif) < TARGET["negatif"]:
            data_negatif.append(r)

        elif rating == 3 and len(data_netral) < TARGET["netral"]:
            data_netral.append(r)

        elif rating in [4, 5] and len(data_positif) < TARGET["positif"]:
            data_positif.append(r)

        # Stop cepat kalau semua sudah penuh
        if (
            len(data_negatif) >= TARGET["negatif"] and
            len(data_netral) >= TARGET["netral"] and
            len(data_positif) >= TARGET["positif"]
        ):
            break

    if continuation_token is None:
        break

    cursor = continuation_token

# Gabungkan semua data
all_reviews = data_negatif + data_netral + data_positif

print("Jumlah Negatif (1–2):", len(data_negatif))
print("Jumlah Netral (3):", len(data_netral))
print("Jumlah Positif (4–5):", len(data_positif))
print("Total data:", len(all_reviews))


Jumlah Negatif (1–2): 2000
Jumlah Netral (3): 2000
Jumlah Positif (4–5): 2400
Total data: 6400


In [2]:
import numpy as np
import pandas as pd

brimo = pd.DataFrame(np.array(all_reviews), columns=['reviews'])

brimo = brimo.join(
    pd.DataFrame(brimo.pop('reviews').tolist())
)

brimo.head(5)

# ====== SIMPAN KE CSV ======
brimo.to_csv("./dataset_mentah/brimo_mentah.csv", index=False, encoding="utf-8")
print("File CSV berhasil dibuat!")


File CSV berhasil dibuat!
